# Dataset Initialization

### Constants

In [2]:
import json
import torch
import re
import os
import math
import numpy as np
import matplotlib.pyplot as plt
from torch.nn import functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM, utils
from datasets import load_dataset

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,garbage_collection_threshold:0.8"

utils.logging.set_verbosity_error()  # Suppress standard warnings

models = [
        # 'meta-llama/Llama-3.2-1B', # Llama-3.2-1B (16 layers 32 heads)
        'meta-llama/Meta-Llama-3-8B', # Meta-Llama-3-8B
        'meta-llama/Meta-Llama-3-70B', # Meta-Llama-3-70B
        'meta-llama/Meta-Llama-3-8B-Instruct', # Meta-Llama-3-8B-Instruct (the only model TRAIL tests with)
        'EleutherAI/gpt-j-6b', # GPT-J 
        'meta-llama/Llama-2-13b-chat-hf', # Llama-2, fine-tuned
        'EleutherAI/gpt-neox-20b', # GPT-NeoX
        ]

datasets = {
        1: './data/dataset_alpaca.json',
        2: './data/datasetSimplified_alpaca.json',
        3: './data/dataset_lmsys-chat-1m.json',
        4: 'yahma/alpaca-cleaned'
}

SPECIAL_LABEL = -1
CACHE_DIR = "./.cache/huggingface/datasets"
FEATURE_DIR = "./training_data/features"
METADATA_DIR = "./training_data/metadata"
DS_NAME = datasets[4]

temperatures = [0.1, 0.3, 0.5, 0.9] # low, mid, high creativity
top_k_values = [1, 2, 3] # low, mid, high diversity
repetition_penalties = [1.0, 1.3, 1.5] # low, mid, high coherence
max_new_tokens_values = [100, 300, 500] # low, mid, high length
system_parameters = []
for temp in temperatures:
        for k in top_k_values:
                for rep_pen in repetition_penalties:
                        for max_tok in max_new_tokens_values:
                                system_parameters.append({
                                        'temperature': temp,
                                        'top_k': k,
                                        'repetition_penalty': rep_pen,
                                        'max_new_tokens': max_tok
                                })
print(f"Total number of system parameters: {len(system_parameters)}")

TEMP_SAMP = temperatures[1]
TK_SAMP = top_k_values[2]
REP_PEN_SAMP = repetition_penalties[2]
MAX_TOKEN_LEN_SAMP = max_new_tokens_values[1]

# Parameters
max_new_tokens = MAX_TOKEN_LEN_SAMP # Maximum number of tokens to generate
temperature = TEMP_SAMP  # Lower temperature for more deterministic output
top_k = TK_SAMP  # Increase top_k for more diverse candidates
repetition_penalty = REP_PEN_SAMP  # Increase repetition penalty to reduce repetition

Total number of system parameters: 108


### Dataset Loading

In [50]:
ds = load_dataset(DS_NAME)
dataset = ds['train']

prompt_template = "{instruction}\n\n{input}"  # template for the prompt
prompts = []

for inst, inp in zip(dataset["instruction"], dataset["input"]):
    if inp.strip() == "":  # no input
        prompt = inst
    else:
        prompt = prompt_template.format(instruction=inst, input=inp)
    prompts.append(prompt)

# structure the data
data = {
    "qa_pairs": [
        {"prompt": prompt, "response": output}
        for prompt, output in zip(prompts, dataset["output"])
    ]
}

# print the output
# print(json.dumps(data, indent=2))

### Model Loading

In [ ]:
# Select model
print("Choose the model to test:")
for i, model in enumerate(models, 1):
    print(f" {i}. {model.split('/')[-1]}")
model_choice = int(input(f"Enter model number (1-{len(models)}): "))
if model_choice < 1 or model_choice > len(models):
    raise ValueError(f"Invalid model choice. Please enter a number between 1 and {len(models)}")
model_name = models[model_choice - 1]

tokenizer = AutoTokenizer.from_pretrained(model_name)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    cache_dir=CACHE_DIR,
    device_map="auto",
    attn_implementation="eager"
)

max_seq_len = tokenizer.model_max_length
eos_token_id = tokenizer.eos_token_id
eos_flag=0

### Function for top-k sampling

In [51]:
def sample_top_k(logits, k, temperature):
    logits = logits / temperature
    top_k_logits, top_k_indices = torch.topk(logits, k)
    probs = F.softmax(top_k_logits, dim=-1)
    return top_k_indices[0, torch.multinomial(probs, 1).item()]

### Simply print the outputs

In [ ]:
results = []
for idx in range(50):
    if idx >= len(data['qa_pairs']):
        break
        
    qa_pair = data['qa_pairs'][idx]
    entry = {
        "original_prompt": qa_pair["prompt"],
        "dataset_response": qa_pair["response"],
        "model_response": "",
        "exceeded_max_seq_len": False,
        "total_sequence_length": 0
    }
    
    # Tokenize prompt
    inputs = tokenizer(qa_pair["prompt"], return_tensors="pt")
    input_ids = inputs.input_ids.to(device)
    attention_mask = inputs.attention_mask.to(device)
    
    # Check input length
    if input_ids.shape[1] > max_seq_len:
        entry["exceeded_max_seq_len"] = True
        entry["total_sequence_length"] = input_ids.shape[1]
        results.append(entry)
        continue
    
    # Generate response
    generated_tokens = []
    current_length = input_ids.shape[1]
    exceeded = False
    eos_reached = False
    
    for _ in range(max_new_tokens):
        if current_length >= max_seq_len:
            exceeded = True
            break
            
        with torch.no_grad():
            outputs = model(input_ids, attention_mask=attention_mask)
        
        logits = outputs.logits[:, -1, :]
        
        # Apply repetition penalty
        for token in generated_tokens:
            logits[0, token] /= repetition_penalty
            
        next_token = sample_top_k(logits, top_k, temperature)
        
        if next_token == eos_token_id:
            eos_reached = True
            break
            
        generated_tokens.append(next_token.item())
        input_ids = torch.cat([input_ids, next_token.view(1, 1)], dim=1)
        attention_mask = torch.cat([attention_mask, torch.ones(1, 1).to(device)], dim=1)
        current_length += 1
    
    entry["model_response"] = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    entry["exceeded_max_seq_len"] = exceeded or (current_length > max_seq_len)
    entry["total_sequence_length"] = current_length
    results.append(entry)
    
    print(f"Processed prompt {idx+1}/50")

# Save results
preview_dir = "./preview"
os.makedirs(preview_dir, exist_ok=True)

# Save results in the preview directory
output_file = os.path.join(preview_dir, f"{model_name.split('/')[-1]}_responses.json")
with open(output_file, 'w') as f:
    json.dump(results, f, indent=2)

print(f"Results saved to {output_file}")

### Prompt Engineering Approach

In [4]:
torch.cuda.empty_cache()

In [ ]:
ADDITION_PROMPT = (
    "Before responding to the above instruction, you must predict the length of your response first: "
    "print the estimated number of words in your response in the first line. "
    "Then change to a new line to respond to the instruction mentioned."
)
# ADDITION_PROMPT = (
#     "IMPORTANT: Your response must follow this format exactly!\n"
#     "Line 1: <prediction> (ONLY a number or range, e.g. 50 or 40-60)\n"
#     "Line 2: <empty>\n"
#     "Line 3+: <actual response>\n"
#     "Prediction must be the token count of your response (lines 3+). "
#     "Do NOT include any other text in line 1."
# )

results = []
predictions_list = []  # New list for prediction/actual pairs

for idx in range(5):
    if idx >= len(data['qa_pairs']):
        break
        
    qa_pair = data['qa_pairs'][idx]
    original_prompt = qa_pair["prompt"]
    
    if ADDITION_PROMPT not in qa_pair["prompt"]:
        qa_pair["prompt"] = f"{qa_pair['prompt']}\n{ADDITION_PROMPT}"
        
    entry = {
        "original_prompt": qa_pair["prompt"],
        # "dataset_response": qa_pair["response"],
        "model_response": "",
        "exceeded_max_seq_len": False,
        "total_sequence_length": 0  # Now represents output tokens (total - input)
    }
    
    # Tokenize prompt
    inputs = tokenizer(qa_pair["prompt"], return_tensors="pt")
    input_ids = inputs.input_ids.to(device)
    attention_mask = inputs.attention_mask.to(device)
    input_token_count = input_ids.shape[1]  # Store input length
    
    # Check input length
    if input_ids.shape[1] > max_seq_len:
        entry["exceeded_max_seq_len"] = True
        entry["total_sequence_length"] = input_ids.shape[1]
        results.append(entry)
        continue
    
    # Generate response
    generated_tokens = []
    current_length = input_ids.shape[1]
    exceeded = False
    # eos_reached = False
    
    for _ in range(max_new_tokens):
        if current_length >= max_seq_len:
            exceeded = True
            break
            
        with torch.no_grad():
            outputs = model(input_ids, attention_mask=attention_mask)
        
        logits = outputs.logits[:, -1, :]
        
        # Apply repetition penalty
        for token in generated_tokens:
            logits[0, token] /= repetition_penalty
            
        next_token = sample_top_k(logits, top_k, temperature)
        
        if next_token == eos_token_id:
            # eos_reached = True
            break
            
        generated_tokens.append(next_token.item())
        input_ids = torch.cat([input_ids, next_token.view(1, 1)], dim=1)
        attention_mask = torch.cat([attention_mask, torch.ones(1, 1).to(device)], dim=1)
        current_length += 1
    
    output_token_count = current_length - input_token_count  # Calculate output length
    entry["model_response"] = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    entry["exceeded_max_seq_len"] = exceeded or (current_length > max_seq_len)
    entry["total_sequence_length"] = output_token_count
    
    # Extract prediction from first line
    prediction_line = entry["model_response"].split("\n", 1)[0].strip()
    predictions_list.append({
        "model output length prediction": prediction_line,
        "actual output length": output_token_count
    })

    results.append(entry)
    print(f"Processed prompt {idx+1}/50")

# Create final output structure
output_data = {
    "predictions": predictions_list,
    "detailed_results": results
}

# Save results
preview_dir = "./preview/pe"
os.makedirs(preview_dir, exist_ok=True)

# Save results in the preview directory
output_file = os.path.join(preview_dir, f"{model_name.split('/')[-1]}_responses_pe.json")
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(output_data, f, indent=2, ensure_ascii=False)
    
print(f"Results saved to {output_file}")

Processed prompt 1/50
Processed prompt 2/50
Processed prompt 3/50
Processed prompt 4/50
Processed prompt 5/50
Results saved to ./preview/pe/Meta-Llama-3-70B_responses_pe.json


### Training Data Initialization

In [ ]:
# sample_features = {
#     "temperature": 1.0,
#     "top_k": 1,
#     "repetition_penalty": 1.3,
#     "layer": 5,
#     "head": 12,
#     "attention_weights": [...],
#     "seq_pos": 25,
#     "label": 10
# }

os.makedirs(FEATURE_DIR, exist_ok=True)
os.makedirs(METADATA_DIR, exist_ok=True)

num_layers = model.config.num_layers
num_heads = model.config.num_heads
print(f"Number of layers: {num_layers}")
print(f"Number of heads: {num_heads}")



# --- traverse dataset ---
for qa_idx, qa in enumerate(data['qa_pairs']):
    prompt = qa['prompt']
    target_response = qa.get('response', '')

    # --- initialize generation state ---
    inputs = tokenizer(prompt, return_tensors="pt")
    input_ids = inputs.input_ids.to(device)
    attention_mask = inputs.attention_mask.to(device)
    generated_tokens = []
    eos_encountered = False
    features = []
    seq_poses = []

    # --- generate tokens ---
    step = 0
    while step < max_new_tokens:
        try:
            with torch.no_grad():
                outputs = model(input_ids, attention_mask=attention_mask, output_attentions=True)
        except RuntimeError as e:
            if "CUDA out of memory" in str(e):
                print(f"CUDA OOM at prompt {qa_idx}, skipping...")
                break
            else:
                raise
        
        # --- feature collection ---
        # input_ids.shape[0]: batch size, input_ids.shape[1]: sequence length, input_ids.shape[2]: vocab size
        # outputs.attentions: list of length num_layers, 
        #                     each element is a tensor of shape (batch_size, num_heads, sequence_length, sequence_length)
        current_seq_pos = input_ids.shape[1] - 1 # position before the new token
        seq_poses.append(current_seq_pos)

        # attention weights substraction
        layer_head_weights = []
        for layer_idx in range(num_layers):
            attentions = outputs.attentions[layer_idx] # (1, num_heads, sequence_length, sequence_length)
            head_weights = attentions[0, :, -1, :].cpu().numpy() # (num_heads, sequence_length)
            layer_head_weights.append(head_weights)

        # transfer to 3D numpy array (num_layers, num_heads, sequence_length)
        feature_step = np.stack(layer_head_weights, axis=0) # (num_layers, num_heads, sequence_length)
        features.append(feature_step)

        # --- token generation ---
        logits = outputs.logits[:, -1, :]
        # Apply repetition penalty
        for token in generated_tokens:
            logits[0, token] /= repetition_penalty
        # sample next token
        next_token = sample_top_k(logits, top_k, temperature)

        # check if EOS token is generated
        if next_token == eos_token_id:
            eos_encountered = True
            generated_tokens.append(next_token.item())
            break

        # update input_ids and attention_mask
        input_ids = torch.cat([input_ids, next_token.view(1, 1)], dim=1)
        attention_mask = torch.cat([attention_mask, torch.ones(1, 1).to(device)], dim=1)
        generated_tokens.append(next_token.item())
        step += 1

    # --- calculate the tags ---
    total_generated_tokens = len(generated_tokens)
    lables = []
    for pos in seq_poses:
        if eos_encountered:
            remaining_tokens = max(0, total_generated_tokens - pos - 1)
        else:
            remaining_tokens = SPECIAL_LABEL
        lables.append(remaining_tokens)

    # --- save the features and metadata ---
    if len(features) > 0:
        # convert to numpy array with compressed format
        feature_array = np.stack(features, axis=0) # (num_steps, num_layers, num_heads, sequence_length)
        model_name_safe = model_name.replace('/', '_') # replace '/' with '_' in model name
        np.savez_compressed(
            os.path.join(FEATURE_DIR, f"{model_name_safe}_{qa_idx}.npz"), 
            features=feature_array.astype(np.float16),
            seq_poses=np.array(seq_poses),
            lables=np.array(lables)
        )

        # save metadata
        metadata = {
            "prompt": prompt,
            "target_response": target_response,
            "generated_response": tokenizer.decode(generated_tokens),
            "total_tokens": total_generated_tokens,
            "eos_encountered": eos_encountered,
            "exceed_max_len": total_generated_tokens >= max_new_tokens,
        }
        with open(os.path.join(METADATA_DIR, f"{model_name_safe}_{qa_idx}.json"), 'w') as f:
            json.dump(metadata, f, indent=2)

    # release memory
    torch.cuda.empty_cache()

# # --- traverse dataset ---
# for qa_idx, qa in enumerate(data['qa_pairs']):
#     prompt = qa['prompt']
#     target_response = qa.get('response', '')

#     # --- initialize generation state ---
#     inputs = tokenizer(prompt, return_tensors="pt")
#     input_ids = inputs.input_ids.to(device)
#     attention_mask = inputs.attention_mask.to(device)
#     generated_tokens = []
#     eos_encountered = False
#     features = []
#     seq_poses = []

#     # --- generate tokens ---
#     step = 0
#     while step < max_new_tokens:
#         try:
#             with torch.no_grad():
#                 outputs = model(input_ids, attention_mask=attention_mask, output_attentions=True)
#         except RuntimeError as e:
#             if "CUDA out of memory" in str(e):
#                 print(f"CUDA OOM at prompt {qa_idx}, skipping...")
#                 break
#             else:
#                 raise
        
#         # --- feature collection ---
#         # input_ids.shape[0]: batch size, input_ids.shape[1]: sequence length, input_ids.shape[2]: vocab size
#         # outputs.attentions: list of length num_layers, 
#         #                     each element is a tensor of shape (batch_size, num_heads, sequence_length, sequence_length)
#         current_seq_pos = input_ids.shape[1] - 1 # position before the new token
#         seq_poses.append(current_seq_pos)

#         # attention weights substraction
#         layer_head_weights = []
#         for layer_idx in range(num_layers):
#             attentions = outputs.attentions[layer_idx] # (1, num_heads, sequence_length, sequence_length)
#             head_weights = attentions[0, :, -1, :].cpu().numpy() # (num_heads, sequence_length)
#             layer_head_weights.append(head_weights)

#         # transfer to 3D numpy array (num_layers, num_heads, sequence_length)
#         feature_step = np.stack(layer_head_weights, axis=0) # (num_layers, num_heads, sequence_length)
#         features.append(feature_step)

#         # --- token generation ---
#         logits = outputs.logits[:, -1, :]
#         # Apply repetition penalty
#         for token in generated_tokens:
#             logits[0, token] /= repetition_penalty
#         # sample next token
#         next_token = sample_top_k(logits, top_k, temperature)

#         # check if EOS token is generated
#         if next_token == eos_token_id:
#             eos_encountered = True
#             generated_tokens.append(next_token.item())
#             break

#         # update input_ids and attention_mask
#         input_ids = torch.cat([input_ids, next_token.view(1, 1)], dim=1)
#         attention_mask = torch.cat([attention_mask, torch.ones(1, 1).to(device)], dim=1)
#         generated_tokens.append(next_token.item())
#         step += 1

#     # --- calculate the tags ---
#     total_generated_tokens = len(generated_tokens)
#     lables = []
#     for pos in seq_poses:
#         if eos_encountered:
#             remaining_tokens = max(0, total_generated_tokens - pos - 1)
#         else:
#             remaining_tokens = SPECIAL_LABEL
#         lables.append(remaining_tokens)

#     # --- save the features and metadata ---
#     if len(features) > 0:
#         # convert to numpy array with compressed format
#         feature_array = np.stack(features, axis=0) # (num_steps, num_layers, num_heads, sequence_length)
#         model_name_safe = model_name.replace('/', '_') # replace '/' with '_' in model name
#         np.savez_compressed(
#             os.path.join(FEATURE_DIR, f"{model_name_safe}_{qa_idx}.npz"), 
#             features=feature_array.astype(np.float16),
#             seq_poses=np.array(seq_poses),
#             lables=np.array(lables)
#         )

#         # save metadata
#         metadata = {
#             "prompt": prompt,
#             "target_response": target_response,
#             "generated_response": tokenizer.decode(generated_tokens),
#             "total_tokens": total_generated_tokens,
#             "eos_encountered": eos_encountered,
#             "exceed_max_len": total_generated_tokens >= max_new_tokens,
#         }
#         with open(os.path.join(METADATA_DIR, f"{model_name_safe}_{qa_idx}.json"), 'w') as f:
#             json.dump(metadata, f, indent=2)

#     # release memory
#     torch.cuda.empty_cache()